In [1]:
import pandas as pd, numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix)

import sys
sys.path.insert(0, '../')
from util import *
from models.cnn_lstm import CNN_LSTM_v1, CNN_LSTM_v2

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

SCENARIOS = [f"../CSV_Files/scene{i+1}/" for i in range(6)]

MODEL_CLS = CNN_LSTM_v2      # or CNN_LSTM_v1
MODEL_NAME = MODEL_CLS.__name__


device: NVIDIA A30


In [2]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:,  -1].astype("int64"))
    return X, y


def load_scenario(scenario_dir):
    train = pd.read_csv(scenario_dir + "train.csv").to_numpy()
    val   = pd.read_csv(scenario_dir + "val.csv").to_numpy()
    test  = pd.read_csv(scenario_dir + "test.csv").to_numpy()
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train, val, test)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)


def make_model(model_cls, device, seed=0, **kwargs):
    """Fresh CNN-LSTM with Xavier init for Conv/Linear; default PyTorch init
    for LSTM (MATLAB's Glorot applies to Conv and FC, LSTM uses its own)."""
    torch.manual_seed(seed)
    model = model_cls(n_classes=8, **kwargs).to(device)
    for m in model.modules():
        if isinstance(m, nn.Conv1d):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=0):
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X_tr), generator=g)
    X_tr_s, y_tr_s = X_tr[perm], y_tr[perm]
    train_loader = DataLoader(TensorDataset(X_tr_s, y_tr_s),
                              batch_size=batch_size, shuffle=False)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss    += criterion(logits, yb).item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(yb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)

def run_scenario(scenario_idx, scenario_dir, model_cls, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(scenario_dir)
    train_loader, val_loader, test_loader = make_loaders(
        X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=seed)

    model = make_model(model_cls, device, seed=seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr,
                           betas=(0.9, 0.999), eps=1e-8,
                           weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    # ---- compute metrics setup ----
    n_params, params_by_type = count_parameters(model)
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({model_cls.__name__}) ===")
        print(f"  params: {n_params:,}  breakdown: {params_by_type}")

    # ---- TRAINING with timer ----
    with Timer(device) as train_timer:
        for ep in range(1, epochs + 1):
            tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                              criterion, optimizer, device)
            va_loss, va_acc = evaluate_loader(model, val_loader,
                                              criterion, device)
            scheduler.step()
            if verbose and (ep == 1 or ep % 10 == 0 or ep == epochs):
                print(f"  ep {ep:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f}"
                      f" | val loss {va_loss:.4f} acc {va_acc:.3f}")

    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- INFERENCE timing ----
    X_te_dev = X_te.to(device)
    def _predict(X):
        model.eval()
        with torch.no_grad():
            return model(X).argmax(1)
    inf_stats = measure_inference_time(_predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- TEST accuracy ----
    y_true, y_pred = predict_all(model, test_loader, device)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: train={train_timer.elapsed:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample  "
              f"peak_mem={peak_mem_mb:.1f}MB")

    return {
        "scenario":    scenario_idx,
        "model":       model_cls.__name__,
        "n_train":     len(X_tr),
        "accuracy":    acc,
        "precision":   p,
        "recall":      r,
        "f1":          f,
        "confusion":   cm,
        # ---- new compute fields ----
        "n_params":    n_params,
        "train_sec":   round(train_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }


In [3]:
results = []
for i, sc_dir in enumerate(SCENARIOS, start=1):
    r = run_scenario(i, sc_dir, MODEL_CLS, device,
                     epochs=70, lr=1e-2, seed=0)
    results.append(r)


=== Scenario 1 (CNN_LSTM_v2) ===
  params: 6,536  breakdown: {'conv1': 36, 'bn1': 24, 'conv2': 300, 'bn2': 24, 'lstm': 5888, 'fc': 264}
  ep   1 | train loss 1.5789 acc 0.368 | val loss 1.0606 acc 0.550
  ep  10 | train loss 0.5138 acc 0.757 | val loss 0.5273 acc 0.741
  ep  20 | train loss 0.3970 acc 0.821 | val loss 0.4563 acc 0.782
  ep  30 | train loss 0.2753 acc 0.886 | val loss 0.6876 acc 0.751
  ep  40 | train loss 0.2227 acc 0.909 | val loss 0.6639 acc 0.777
  ep  50 | train loss 0.1554 acc 0.942 | val loss 0.5184 acc 0.826
  ep  60 | train loss 0.1262 acc 0.955 | val loss 0.5039 acc 0.846
  ep  70 | train loss 0.1091 acc 0.963 | val loss 0.2765 acc 0.896
  TEST: acc=0.9006  P=0.9105  R=0.9006  F1=0.9001
  COMPUTE: train=79.5s  inf=0.001ms/sample  peak_mem=25.7MB

=== Scenario 2 (CNN_LSTM_v2) ===
  params: 6,536  breakdown: {'conv1': 36, 'bn1': 24, 'conv2': 300, 'bn2': 24, 'lstm': 5888, 'fc': 264}
  ep   1 | train loss 1.7907 acc 0.274 | val loss 1.5283 acc 0.347
  ep  10 | tr

In [4]:
summary = pd.DataFrame([
    {k: r[k] for k in ("scenario", "model", "n_train",
                       "accuracy", "precision", "recall", "f1",
                       "n_params", "train_sec", "inf_ms_per_sample",
                       "peak_mem_mb")}
    for r in results
])
cols_round = ["accuracy", "precision", "recall", "f1"]
summary[cols_round] = summary[cols_round].round(4)
print(summary.to_string(index=False))

summary.to_csv(f"{MODEL_NAME.lower()}_results_by_scenario.csv", index=False)


 scenario       model  n_train  accuracy  precision  recall     f1  n_params  train_sec  inf_ms_per_sample  peak_mem_mb
        1 CNN_LSTM_v2     7858    0.9006     0.9105  0.9006 0.9001      6536      79.47             0.0005         25.7
        2 CNN_LSTM_v2     3939    0.8150     0.8274  0.8150 0.8106      6536      38.39             0.0005         25.7
        3 CNN_LSTM_v2      786    0.7006     0.7129  0.7006 0.6969      6536       9.63             0.0005         25.7
        4 CNN_LSTM_v2      391    0.7438     0.7494  0.7438 0.7389      6536       8.02             0.0005         25.7
        5 CNN_LSTM_v2      238    0.7538     0.7560  0.7538 0.7544      6536       6.38             0.0005         25.7
        6 CNN_LSTM_v2       77    0.6181     0.6085  0.6181 0.6094      6536       5.25             0.0005         25.7


In [5]:
for r in results:
    print(f"\nScenario {r['scenario']}  ({r['model']}, "
          f"n_train={r['n_train']}, acc={r['accuracy']:.3f})")
    print(r["confusion"])



Scenario 1  (CNN_LSTM_v2, n_train=7858, acc=0.901)
[[193   2   0   5   0   0   0   0]
 [  0 176   0   0   0   7   0  17]
 [  0   0 195   0   0   5   0   0]
 [ 68   2   0 130   0   0   0   0]
 [  0   0   0   0 200   0   0   0]
 [  0   1   4   0   0 183   0  12]
 [  6   0   0   7   1   0 186   0]
 [  0   3   1   0   0  18   0 178]]

Scenario 2  (CNN_LSTM_v2, n_train=3939, acc=0.815)
[[162  15   0  20   2   1   0   0]
 [  1 168   0   0   0   6   0  25]
 [  0   0 200   0   0   0   0   0]
 [101  10   0  89   0   0   0   0]
 [  0   0   0   0 200   0   0   0]
 [  0   0  24   0   0 172   0   4]
 [ 14   0   0   6   0   0 180   0]
 [  0   1  12   0   0  54   0 133]]

Scenario 3  (CNN_LSTM_v2, n_train=786, acc=0.701)
[[138  12   0  49   0   0   0   1]
 [  0 124   0   1   0   1   0  74]
 [  0   0 199   0   0   1   0   0]
 [128   4   0  66   1   0   1   0]
 [  0   0   0   0 200   0   0   0]
 [  1   0  57   0   0 131   0  11]
 [  9   0   0  29   0   0 162   0]
 [  0   1  18   0   1  79   0 101]]

S